# Statcast 2023-2024 데이터 탐색

SmartPitch MDP 전이확률 모델 비교를 위한 데이터 탐색 노트북.

## 목적

이 노트북은 다운로드된 Statcast 2023-2024 데이터를 탐색하여:
1. 데이터 규모와 품질 파악
2. 카테고리 변수의 unique 값 확인 (모델 라벨 매핑용)
3. 결측치 패턴 분석 (deprecated 컬럼 식별)
4. 타자별 pitch 수 분포 (Sliding window 가능성 검증)
5. 시즌 간 분포 차이 비교

이를 통해 Phase 3.2 (Feature 매핑) 단계에서 정확한 결정을 내릴 수 있도록 한다.

## 1. 라이브러리 import 및 시각화 설정

- pandas: 데이터프레임 처리
- numpy: 수치 연산
- matplotlib, seaborn: 시각화
- 그래프 기본 크기와 폰트 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")  # seaborn 기본 테마 - 격자 배경
plt.rcParams["figure.figsize"] = (12, 6)  # 기본 그래프 크기 (가로 12, 세로 6)
plt.rcParams["font.size"] = 11  # 기본 폰트 크기

## 2. 데이터 로드 및 결합

2023, 2024 두 시즌의 Parquet 파일을 로드하고 하나의 DataFrame으로 결합한다.
- `season` 컬럼 추가: 추후 시즌별 비교 시 사용
- `game_date`를 datetime으로 변환

In [ ]:
# 각 시즌 데이터 로드 (Parquet 형식 - CSV보다 빠르고 타입 보존)
df_2023 = pd.read_parquet("../data/raw/2023/statcast_2023.parquet")
df_2024 = pd.read_parquet("../data/raw/2024/statcast_2024.parquet")

# 두 시즌을 하나로 합침. ignore_index로 0부터 새 인덱스
df = pd.concat([df_2023, df_2024], ignore_index=True)

# game_date 컬럼을 datetime 타입으로 변환 (월/연도 추출 위해)
df["game_date"] = pd.to_datetime(df["game_date"])

# datetime의 .dt 접근자로 연도 추출 → season 컬럼 추가
df["season"] = df["game_date"].dt.year

print(f"2023: {len(df_2023):,} rows")
print(f"2024: {len(df_2024):,} rows")
print(f"Combined: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 3. 기본 정보 확인

DataFrame의 크기, 컬럼별 데이터 타입, 메모리 사용량을 확인한다.
- shape: (행 수, 열 수)
- dtypes value_counts: 데이터 타입별 컬럼 수
- memory_usage: 실제 메모리 사용량 (`deep=True`로 object 타입까지 정확히)

In [ ]:
# 전체 행 수와 열 수
print(f"Shape: {df.shape}")

# 각 dtype별 몇 개 컬럼이 있는지
print(f"\nDtype distribution:")
print(df.dtypes.value_counts())

# object 컬럼(문자열 등)까지 실제 메모리 측정
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 4. 결측치 분포 분석

컬럼별 결측 비율을 시각화한다.
- **100% 결측** = deprecated 컬럼 (Statcast가 더 이상 업데이트하지 않음)
- **~67% 결측** = InPlay 전용 feature (launch_speed 등) — 정상
- **~2.4% 결측** = 측정 오류 (해당 row 제거 대상)

In [ ]:
# 각 컬럼의 결측 비율 (%)
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_cols = null_pct[null_pct > 0]  # 결측 있는 컬럼만

# 100% 결측 컬럼 = 사용 불가, 전처리 시 제거
deprecated = null_pct[null_pct >= 99.9].index.tolist()
print(f"Deprecated columns (≥99.9% null): {deprecated}\n")

fig, ax = plt.subplots(figsize=(10, 10))
# 수평 막대 그래프 (컬럼명이 길어서 수평이 보기 좋음)
null_cols.head(40).plot.barh(ax=ax, color="coral")
ax.set_xlabel("Missing ratio (%)")
ax.set_title(f"Column null ratio (top 40 of {len(null_cols)} with nulls)")
ax.invert_yaxis()  # 큰 값이 위에 오도록 y축 뒤집기
plt.tight_layout()

## 5. 카테고리 변수 unique 값 분석

다음 컬럼들의 고유값과 빈도를 확인한다:
- **pitch_type**: 구종 (FF/SI/SL/CH/CU 등) — Model 입력 feature
- **events**: 타석 결과 (single/double/strikeout 등) — Model C의 10-class 라벨용
- **description**: 투구 결과 상세 (called_strike/ball 등) — Model B의 4-class 라벨용
- **type**: 투구 결과 단순 (B/S/X) — Model B 검증용
- **zone**: 투구 위치 (1~14) — Model 입력 feature
- **stand**: 타자 좌우 — Model 입력 feature
- **p_throws**: 투수 좌우 — Model 입력 feature

이 결과가 Phase 3.2의 라벨 매핑 함수 설계의 기준이 됨.

In [ ]:
cat_cols = ["pitch_type", "events", "description", "type", "zone", "stand", "p_throws"]

for col in cat_cols:
    if col not in df.columns:
        continue
    # 결측치도 카운트에 포함 (NaN을 별도 카테고리로)
    vc = df[col].value_counts(dropna=False)
    n_unique = df[col].nunique()  # 고유값 개수 (NaN 제외)
    null_n = df[col].isnull().sum()
    print(f"\n{'='*50}")
    print(f"{col} — {n_unique} unique, {null_n:,} nulls ({null_n/len(df)*100:.1f}%)")
    print(vc.to_string())

## 6. 타자별 pitch 수 분포

Model C(Transformer)는 한 타자의 연속 400 pitches를 입력으로 받음.
따라서 400+ pitches 가진 타자가 충분한지 확인이 중요.
- **빨간 선 (400)**: Model C의 sliding window 최소 요구치
- **주황 선 (200)**: 200~400은 패딩 또는 별도 처리 고려

In [ ]:
# 타자별 pitch 수 집계
batter_counts = df.groupby("batter").size().rename("n_pitches")  # Series 이름 부여

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# === 전체 분포 ===
batter_counts.hist(bins=80, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(400, color="red", ls="--", lw=1.5, label="400 pitches")  # 임계값 표시
axes[0].axvline(200, color="orange", ls="--", lw=1.5, label="200 pitches")
axes[0].set_xlabel("Pitches per batter")
axes[0].set_ylabel("Count")
axes[0].set_title("All batters")
axes[0].legend()

# === 400+ 확대 ===
batter_400 = batter_counts[batter_counts >= 400]
batter_400.hist(bins=40, ax=axes[1], color="seagreen", edgecolor="white")
axes[1].set_xlabel("Pitches per batter")
axes[1].set_title(f"Batters with 400+ pitches (n={len(batter_400)})")

plt.tight_layout()

# 조건 충족하는 타자 수 카운트 (Boolean 합계)
print(f"Total batters: {len(batter_counts):,}")
print(f"Mean: {batter_counts.mean():.1f}, Median: {batter_counts.median():.1f}")
print(f"400+ pitches: {(batter_counts >= 400).sum()}")
print(f"200-400 pitches: {((batter_counts >= 200) & (batter_counts < 400)).sum()}")
print(f"<200 pitches: {(batter_counts < 200).sum()}")

## 7. 시즌별 분포 비교 (2023 vs 2024)

두 시즌의 분포가 비슷한지 확인.
- **비슷하면**: Train/Test를 시즌으로 나눠도 일반화 잘 됨
- **차이 크면**: 룰 변경, 트렌드 변화 가능성 (예: Pitch Clock 영향)
- **월별 pitch 수**: 시즌 시작/끝 시기 확인

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# === pitch_type 분포 ===
ct = pd.crosstab(df["pitch_type"], df["season"])  # 두 변수의 교차표
ct_top = ct.loc[ct.sum(axis=1).nlargest(10).index]  # 상위 10개 pitch type만
ct_top.plot.bar(ax=axes[0])
axes[0].set_title("Pitch type by season (top 10)")
axes[0].set_ylabel("Count")

# === type (B/S/X) 비율 ===
# 열 단위로 정규화 (각 시즌 비율로 변환)
ct_type = pd.crosstab(df["type"], df["season"], normalize="columns") * 100
ct_type.plot.bar(ax=axes[1])
axes[1].set_title("Outcome type ratio by season (%)")
axes[1].set_ylabel("Percentage")

# === 월별 pitch 수 ===
df["month"] = df["game_date"].dt.month
monthly = df.groupby(["season", "month"]).size().unstack("season")
monthly.plot(ax=axes[2], marker="o")
axes[2].set_title("Monthly pitch counts")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Pitches")

plt.tight_layout()

## 8. 핵심 연속 feature 분포

모델 입력으로 사용할 주요 연속 변수의 분포 확인.
- **정규분포**: 표준화(StandardScaler) 적합
- **치우친 분포**: 로버스트 스케일링 또는 변환 고려
- **다봉(multimodal)**: 카테고리(예: 구종)에 따라 다른 분포일 가능성

In [ ]:
# 분석할 8개 핵심 feature 리스트
num_cols = [
    "release_speed", "release_spin_rate", "pfx_x", "pfx_z",
    "plate_x", "plate_z", "launch_speed", "launch_angle",
]

# 2행 4열 = 8개 subplot 생성
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

# axes.flat: 2D axes 배열을 1D로 평탄화 (zip 사용 위해)
for ax, col in zip(axes.flat, num_cols):
    data = df[col].dropna()
    ax.hist(data, bins=60, alpha=0.7, color="steelblue", edgecolor="white")  # alpha=0.7 투명도
    ax.axvline(data.mean(), color="red", ls="--", lw=1, label=f"μ={data.mean():.1f}")  # 평균값 표시
    ax.set_title(f"{col}\n(null {df[col].isnull().mean()*100:.1f}%)")  # 결측 비율 제목에 표시
    ax.legend(fontsize=8)

plt.suptitle("Core feature distributions", fontsize=14, y=1.01)
plt.tight_layout()